# Camada Bronze: Ingestão de Dados (Landing to Bronze)

Este notebook é responsável pela primeira etapa do nosso Data Lakehouse. O objetivo aqui é extrair os dados brutos e carregá-los em tabelas Delta, garantindo a rastreabilidade através de metadados.

**Principais Atividades:**
* **Criação do Banco de Dados:** Inicialização do schema `bronze`.
* **Ingestão de Arquivos CSV:** Leitura de arquivos brutos do Volume (Landing Zone) e conversão para o formato Delta.
* **Consumo de API Externa:** Extração de cotações do Dólar diretamente do Banco Central (Olinda BCB) para cálculos financeiros posteriores.
* **Linhagem de Dados:** Adição da coluna `timestamp_ingestion` em todas as tabelas.

###  Ingestão de Arquivos Locais (Olist Dataset)
Este bloco percorre o dicionário de mapeamento, lê os arquivos CSV armazenados no Volume e os salva como tabelas Delta no banco `bronze`. O processo utiliza a inferência de schema e adiciona o carimbo de tempo da carga.

In [0]:
# 1. Criar o banco de dados da camada Bronze
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

from pyspark.sql.functions import current_timestamp

# Defina o caminho do seu volume (ajuste se o nome for diferente)
# O padrão é /Volumes/workspace/default/bronze/
volume_path = "/Volumes/workspace/bronze/dados_brutos/"

# Lista de mapeamento (Arquivo CSV -> Nome da Tabela) conforme o PDF
mapeamento = {
    "olist_customers_dataset.csv": "tb_customers",
    "olist_geolocation_dataset.csv": "tb_geolocalizacao",
    "olist_order_items_dataset.csv": "tb_order_items",
    "olist_order_payments_dataset.csv": "tb_order_payments",
    "olist_order_reviews_dataset.csv": "tb_order_reviews",
    "olist_orders_dataset.csv": "tb_orders",
    "olist_products_dataset.csv": "tb_products",
    "olist_sellers_dataset.csv": "tb_sellers",
    "product_category_name_translation.csv": "tb_product_category_name_translation"
}

# Loop para ler cada arquivo e salvar como tabela
for arquivo, tabela in mapeamento.items():
    path = f"{volume_path}{arquivo}"
    
    # Ler o CSV
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(path)
    
    # Adicionar a coluna timestamp_ingestion exigida
    df_bronze = df.withColumn("timestamp_ingestion", current_timestamp())
    
    # Salvar no banco de dados bronze
    df_bronze.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"bronze.{tabela}")
    
    print(f"Tabela bronze.{tabela} criada com sucesso!")

Tabela bronze.tb_customers criada com sucesso!
Tabela bronze.tb_geolocalizacao criada com sucesso!
Tabela bronze.tb_order_items criada com sucesso!
Tabela bronze.tb_order_payments criada com sucesso!
Tabela bronze.tb_order_reviews criada com sucesso!
Tabela bronze.tb_orders criada com sucesso!
Tabela bronze.tb_products criada com sucesso!
Tabela bronze.tb_sellers criada com sucesso!
Tabela bronze.tb_product_category_name_translation criada com sucesso!


###  Integração com API de Cotação (Banco Central)
Para converter os valores de venda de BRL para USD na camada Silver, este bloco consome a API do Banco Central. Ele extrai as cotações de compra e venda de um período específico e armazena os dados brutos na tabela `bronze.tb_cotacao_dolar`.

In [0]:
import requests
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# Parâmetros (Exemplo: Janeiro de 2026)
data_inicio = '01-01-2026'
data_fim = '03-31-2026'

# O 'r' antes das aspas resolve o aviso do \$
url = r"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial='{data_inicio}',dataFinalCotacao='{data_fim}')?\$select=dataHoraCotacao,cotacaoCompra&\$format=json".format(data_inicio=data_inicio, data_fim=data_fim)

response = requests.get(url)
data = response.json()['value']

# Criar DataFrame da API
df_dolar = spark.createDataFrame(data)
df_dolar = df_dolar.withColumn("timestamp_ingestion", current_timestamp())

# Salvar
df_dolar.write.format("delta").mode("overwrite").saveAsTable("bronze.tb_cotacao_dolar")
print("Tabela bronze.tb_cotacao_dolar criada!")

Tabela bronze.tb_cotacao_dolar criada!
